# Python Web Scraping
This beginner-friendly Python tutorial demonstrates how to fetch web pages from URLs, parse HTML content using BeautifulSoup, and transform extracted data into a structured Pandas DataFrame for analysis.

By the end of this notebook, you should be able to:

- Send HTTP requests to URLs with Python
- Handle HTTP responses safely
- Parse HTML documents with BeautifulSoup
- Extract structured information from unstructured HTML
- Store and analyse results with Pandas DataFrames
# Why Should I Learn This?
These skills and tools are applicable across many fields:

- Data Analysis and Data Science
    - Pandas is a core data analysis tool, often industry standard for labelled, human-readable data
    - Web data is often messy, and needs to be parsed and cleaned to form a useful dataset
- Automation and Scripting
    - Web requests and HTML parsing are useful for automation tasks that involve retrieving, interpreting and acting upon web-based information
    - Like writing a program that sends you a notification when something goes on sale online!
- Backend and General Software Engineering
    - Core concepts like HTTP requests, error handling and data-transfer pipelines are present across the breadth of software engineering
# What Are We Scraping?
For this tutorial, we will be setting up a scraper for Durham E-theses, an online repository of research papers submitted by Durham University students. By the end, we will be able to sample a subset of theses and represent it as a Pandas DataFrame, then see what we can find out about the repository and its contents!
# Setting Up Our Tools
First, we need to install and import the libraries we're going to be using: 

In [ ]:
%pip install bs4
%pip install pandas

In [2]:
import requests # For making HTTP requests and catching responses
from bs4 import BeautifulSoup # For parsing HTTP responses
import pandas as pd # For storing data in structured DataFrames
import time # We'll need this later!

# Checking Out The Website
Before we start writing any code, and **especially** before we start automating any HTTP requests, we need to make sure we understand the website we are scraping. We also need to check what we can **ethically** scrape.

Head to https://etheses.dur.ac.uk/ and have a look around, and keep an eye on the URL at the top as you move between pages. Once you find a thesis, you'll notice the URL is something like https://etheses.dur.ac.uk/453/ or https://etheses.dur.ac.uk/2124/, and that each web page for a thesis has a set structure.
## Inspect Element
Once you're on a page for a thesis, hit Ctrl+Shift+I (Or Command+Option+I on Mac). This lets you view the HTML response your browser is outputting! More particularly, you can view the DOM (Domain Object Model), a structure containing elements like buttons, labels, and titles that form the contents of the webpage.

The DOM allows for the contents of each element to include other elements, forming a tree of subelements and their subelements and so-on. We identify elements by their:

- **id** - A **unique** identifier
- **class** - An identifier that can be shared with other elements
- **type** - An identifier that gives the element certain properties and defines what it can do

Consider the following structure:

```
<div id="panel">
    <p id="panel_info">
        This is the control panel, try pressing these buttons below:
    </p>
    <button id="start_button" class="panel_item">
        PRESS ME TO START
    </button>
    <button id="pause_button" class="panel_item">
        PRESS ME TO PAUSE
    </button>
</div>
```

We can see that the "panel" div contains three subelements, a **paragraph** (denoted "p"), and two **buttons** that share a class but use separate ids. We call `start_button` and `pause_button` **siblings**, as they are both **children** of the **parent** `panel`. The DOM has order from top to bottom, meaning that the **next** sibling of `panel_info` is `start_button` and the next sibling of `start_button` is `pause_button`. These terms can be used to define traversal up, down, and along a HTML structure. 

Try and navigate the DOM of the page you are on to find the elements containing the title, abstract, and other useful data of the thesis, you should find these in:

```
<body>
 └> <div id="page_wrapper">
     └> <div class="prel">
         └> <div id="main">
             └> <div class="maincontent">
                 └> <div class="contentblock">
                     └> <div id="ep_main">
                         └> <div class="ep_tm_main">
                             ├> <h1 class="ep_tm_pagetitle">
                             |   └> The title is here!
                             └> <div class="ep_summary_content">
                                 └> <div class="ep_summary_content_main">
                                     ├> <div class="ep_block">
                                     |   └> <p>
                                     |       └> Here is the abstract!
                                     └> <table class="ep_block>
                                         └> Here is a table with extra data!
```

Now we've found the abstract, title, and metadata, its clear that we can get a python script to navigate this tree like we just did.
## Understanding Scraping Permissions
Arguably the most important part of web-scraping is checking to see if the website owner allows us to automatically sift through their website. Owners may disallow or restrict this for the following reasons:

- To minimise server load
    - A web scraper can generate a large number of requests in a small time-frame
    - This can slow down an unprepared website, or even cause outages for users
- To protect intellectual property and content ownership
    - Website content may be copyrighted or licensed
    - Scraping can enable unauthorised reuse and distribution
- Availability of Official API's
    - A website may offer an API for controlled access to their data
    - Often to ensure a predictable load and control what is returned

The almost universal source of rules for what an automated scraper can do is the "robots.txt", a text file detailing permitted and forbidden actions scrapers can take.
### Interpreting the robots.txt
Let's pick apart the following example:

```
User-agent: *
Disallow: /users/
Disallow: /posts/
Allow: /posts/public
Crawl-delay: 5
```

- "User-agent:" Specifies which crawlers the following rules apply to (usually * for all) 
- "Disallow:" Blocks access to specific URLs or directories
- "Allow:" Overrides "Disallow:" to specify permitted access for sub-paths
- "Crawl-delay:" A time period (in seconds) that a scraper should wait between request

So we can see that this robots.txt applies to ALL scrapers, and disallows scraping of anything in:

- "example.com/users/"
- "example.com/posts/" UNLESS it is in "example.com/posts/public"

Now go to https://etheses.dur.ac.uk/robots.txt and have a look at the rules we need to abide by, it should read:

```
User-agent: *
Disallow: /cgi/
Disallow: /10862/
```

This tells us that we are fully allowed to scrape the URLs with thesis information, EXCEPT for https://etheses.dur.ac.uk/10862. There is no crawl-delay, but we will implement a short delay to be respectful regardless (that's why we imported the time package earlier!).

Now we have navigated the DOM and checked for ethical constraints, we can start writing our scraper!

# Let's Actually Code!
Now we are ready to put our knowledge of HTML structures and our investigation of Durham E-theses towards a working Python program. By the end of this section you will have:

- Made a HTTP request and stored the response
- Parsed the response with BeautifulSoup
- Stored the response in a Pandas DataFrame 
## Call and Response
The following code constructs a URL for a particular theses and makes a HTTP request, printing the response body. Give it a run!

In [3]:
HEADERS = {"User-Agent": "DUWiT Hacks 2026 HackPack"} # Gives our scraper a User-Agent name
BASE_URL = "https://etheses.dur.ac.uk" # We can append the rest of the URL to this base

# Construct our URL
test_id = 451
test_url = f"{BASE_URL}/{test_id}" # This will be "https://etheses.dur.ac.uk/451"

try: # A try-except makes sure that we can execute code when something in the "try" clause fails

    # Use the URL to make a HTTP request with our HEADERS and a timeout of 10 seconds
    response = requests.get(test_url, headers=HEADERS, timeout=10)

    # Print the first 2500 characters of the response
    print(response.text[:2500]) 
except requests.RequestException as e:
    print(f"Error fetching {url}: {e}")

<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN"
"http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">
<html>
  <head>
    <meta name="google-site-verification" content="OpxnBx6TwYG7Vewk9h9Ryn_x2z65ACzWKK2ZsY19_DQ" />
    <meta name="google-site-verification" content="vsZHvPXzU4tukK2XncNEfSIWVOR53S-8kbabL_Aa-S4" />
    <title>The Renewal of Song: Metalepsis and the Christological Revision of Psalmody in Paul - Durham e-Theses</title>
    <script type="text/javascript" src="/secure/javascript/auto.js"><!-- padder --></script> 
    <style type="text/css" media="screen">@import url(/secure/style/auto.css);</style>
    <style type="text/css" media="print">@import url(/secure/style/print.css);</style>
    <link rel="icon" href="/secure/favicon.ico" type="image/x-icon" />
    <link rel="shortcut icon" href="/secure/favicon.ico" type="image/x-icon" />
    <link rel="Top" href="http://etheses.dur.ac.uk/" />
    <link rel="Search" href="http://etheses.dur.ac.uk/cgi/search" />

In [ ]:
Now we have the response, we can get started on parsing and extracting the contents with BeautifulSoup

## Soup Time

To use BeautifulSoup, you first make and assign a **soup** object to the contents of a HTTP response: 

In [4]:
soup = BeautifulSoup(response.text, "html.parser")

Now we can easily apply `soup` to find the title and abstract like so:

In [5]:
abstract = ""
title = ""

# The abstract is within a <p> element directly below a <h2> with the text "Abstract"

headers = soup.find_all("h2") # Returns a list of all <h2> elements
for h in headers:
    if h.get_text(strip=True) == "Abstract": # Check if the text is "Abstract"
        # Find the first <p> after the Abstract header
        p = h.find_next_sibling("p") # Get the next sibling with the <p> tag
        if p:
            abstract = p.get_text(strip=True) # strip removes spaces on either side

# The title is within a <h1> element, uniquely with the class "ep_tm_pagetitle"

headers = soup.find_all("h1")
for h in headers:
    if h.has_attr('class'): # Checks if the header has a class
        class_list = h['class'] # An element can have many classes, so this returns a list
        if class_list[0] == 'ep_tm_pagetitle': # Check the first class in class_list
            title = h.get_text(strip=True) 

print(f"Title:\n\n{title}\n")
print(f"Abstract:\n\n{abstract}")

Title:

The Renewal of Song: Metalepsis and the Christological Revision of Psalmody in Paul

Abstract:

The productive yield of Richard Hays’ Echoes of Scripture in the Letters of Paul for the study of Pauline intertextuality has not been matched by adequate reflection on questions of method, particularly on the character of the trope at the heart of the Haysian project: metalepsis, or “echo”.  Nor has sufficient attention been given to the reception of biblical psalmody in Paul, and to the distinctiveness of psalmic discourse in relation to metaleptic process.  This study accordingly attempts a close engagement with biblical psalmody as this appears at selected sites in Romans and 2 Corinthians, focusing on those sites which best demonstrate the distinctive character of psalmody, and so offer to refine an account of metalepsis.  In particular, it examines quotations which are attributed or attributable to David or to Christ, and sites in which psalmody serves to modulate Paul’s discou

Getting the data from the table is slightly more complicated:

In [6]:
# The table is a <table> with class "ep_block"

table = None
tables = soup.find_all("table")
for t in tables:
    if t.has_attr('class'):
        if t['class'][0] == "ep_block":
            table = t 

# Extract rows from table

table_dict = {} # Placing rows in a dictionary (key-value pairs)
for tr in table.find_all("tr"): # each <tr> is a row
    key = tr.find("th").get_text(strip=True)[:-1] # Removes trailing colon
    data = tr.find("td").get_text(strip=True)
    table_dict[key] = data


print(table_dict)

{'Item Type': 'Thesis (Doctoral)', 'Award': 'Doctor of Philosophy', 'Keywords': 'Paul psalmody intertextuality metalepsis echo Christology Hays subjectivity', 'Faculty and Department': 'Faculty of Arts and Humanities > Theology and Religion, Department of', 'Thesis Date': '2010', 'Copyright': 'Copyright of this thesis is held by the author', 'Deposited On': '20 Dec 2010 10:18'}


Just like that we have converted a messy HTML page into clean, consistent data! 

## Automation

What we just did was great, but it would be even better if we could modify our script to automatically sample a set of theses from the website without us having to check the URL first. This would require:

- Randomly generating a URL
- Attempting a HTTP request


In [7]:
import random # Needed for random URL generation

theses = [] # Initial empty thesis list
TARGET_COUNT = 20 # Sample size

def extract_abstract(soup):

    headers = soup.find_all("h2")
    for h in headers:
        if h.get_text(strip=True) == "Abstract":
            # Find the first <p> after the Abstract header
            p = h.find_next_sibling("p")
            if p:
                return p.get_text(strip=True)
    return None

def extract_title(soup):

    headers = soup.find_all("h1")
    for h in headers:
        if h.has_attr('class'):
            if h['class'][0] == 'ep_tm_pagetitle':
                return h.get_text(strip=True)
    return None
    
def extract_table(soup):

    table = None
    tables = soup.find_all("table")
    for t in tables:
        if t.has_attr('class'):
            if t['class'][0] == "ep_block":
                table = t 

    table_dict = {}
    for tr in table.find_all("tr"):
        key = tr.find("th").get_text(strip=True)[:-1]
        data = tr.find("td").get_text(strip=True)
        table_dict[key] = data
    return table_dict

print(f"Beginning scrape for {TARGET_COUNT} theses")

while len(theses) < TARGET_COUNT:

    # pick a random thesis ID
    current_id = 0
    while True:
        current_id = random.randint(0,6000)
        if current_id != 10862: # disallowed in robots.txt
            break

    url = f"https://etheses.dur.ac.uk/{current_id}" # Construct URL by appending the ID

    try:
        response = requests.get(url, headers=HEADERS, timeout=10) # Send a HTTP request and await the response

        if response.status_code != 200: # 200 means a successful return, otherwise try again
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        abstract = extract_abstract(soup)
        title = extract_title(soup)
        table = extract_table(soup)
        if abstract and title and table: # All scraper elements MUST be found
            thesis_entry = {
                "Title": title,
                "Abstract": abstract,
                **table # This unpacks dictionary "table" and stores its fields in thesis_entry 
            }
            theses.append(thesis_entry) # Add new thesis to our list
            print(f"[Y] Collected thesis {len(theses)}/{TARGET_COUNT} from {url}")
        else: # Thesis was missing data
            print(f"[N] Missing data at {url}")
    except requests.RequestException as e: # In the case of request timeout etc
        print(f"[!] Error fetching {url}: {e}")

    time.sleep(1) # Wait for a second before making another request, implementing our own crawl-delay

print(f"\nCollected {len(theses)} theses.")

Beginning scrape for 20 theses
[!] Error fetching https://etheses.dur.ac.uk/178: HTTPSConnectionPool(host='etheses.dur.ac.uk', port=443): Read timed out. (read timeout=10)
[!] Error fetching https://etheses.dur.ac.uk/4669: HTTPSConnectionPool(host='etheses.dur.ac.uk', port=443): Read timed out. (read timeout=10)
[!] Error fetching https://etheses.dur.ac.uk/5331: HTTPSConnectionPool(host='etheses.dur.ac.uk', port=443): Read timed out. (read timeout=10)
[!] Error fetching https://etheses.dur.ac.uk/3925: HTTPSConnectionPool(host='etheses.dur.ac.uk', port=443): Read timed out. (read timeout=10)
[!] Error fetching https://etheses.dur.ac.uk/5340: HTTPSConnectionPool(host='etheses.dur.ac.uk', port=443): Read timed out. (read timeout=10)
[!] Error fetching https://etheses.dur.ac.uk/1367: HTTPSConnectionPool(host='etheses.dur.ac.uk', port=443): Read timed out. (read timeout=10)
[!] Error fetching https://etheses.dur.ac.uk/4359: ('Connection aborted.', ConnectionResetError(104, 'Connection reset

Through error handling and response parsing, we have a robust method of sampling theses from the repository. Now, we will use Pandas DataFrames as a way to store, represent and manipulate our raw thesis data.

# Pandas

Pandas is a Python library that lets us work with **tabular** data (data in a structured format of rows and columns). It provides functionality for tasks like cleaning, filtering and analyising data, as well as computing statistics with minimal code.

## DataFrame

A DataFrame is a table of data comprised of rows and columns, similar to an Excel spreadsheet or SQL table. Currently, a single row of our data would contain the following fields:

```
{
    'Title': 'Web Scraping and How Cool It Is',
    'Abstract': 'Web scraping is super fun and cool. This paper details how to do it and be super employable and tech-savvy',
    'Item Type': 'Thesis (Masters)',
    'Award': 'Master of Jurisprudence',
    'Thesis Date': '2026',
    'Copyright': 'Copyright of this thesis is held by the author',
    'Deposited On': '06 Jan 2026 11:52'
}
```

We can easily convert a list of these dictionaries (sets of key-value pairs) into a DataFrame like so:

In [ ]:
data = [
    {
        'Title': 'Quantum Physics',
        'Abstract': 'This thesis explores quantum entanglement...',
        'Item Type': 'Thesis (Doctoral)',
        'Award': 'Doctor of Physics',
        'Thesis Date': '1996',
        'Copyright': 'Held by author',
        'Deposited On': '09 Oct 2012 11:52'
    },
    {
        'Title': 'Constitutional Law',
        'Abstract': 'An analysis of constitutional law...',
        'Item Type': 'Thesis (Masters)',
        'Award': 'Master of Jurisprudence',
        'Thesis Date': '1997',
        'Copyright': 'Held by author',
        'Deposited On': '10 Oct 2012 14:30'
    },
    {
        'Title': 'Materials Engineering',
        'Abstract': 'A deep dive into materials engineering...',
        'Item Type': 'Thesis (Masters)',
        'Award': 'Master of Engineering',
        'Thesis Date': '2001',
        'Copyright': 'Held by author',
        'Deposited On': '21 Jun 2013 09:30'
    }
]

df = pd.DataFrame(data) # It is as easy as this!
df

Now that we have created our DataFrame, we can start exploring the data.

Let's have a look at the first few rows of the dataset

In [8]:
df = pd.DataFrame(theses)
df.head()

,Title,Abstract,Item Type,Award,Thesis Date,Copyright,Deposited On,Keywords,Faculty and Department
0,Runtime visualisation of object-oriented software,"Software is a complex and invisible entity, ye...",Thesis (Doctoral),Doctor of Philosophy,2003,Copyright of this thesis is held by the author,26 Jun 2012 15:21,NaN,NaN
1,"Redefining the Self:Life Writing, Fairy Tale a...",This thesis primarily focuses on Amélie Nothom...,Thesis (Masters),Master of Arts,2012,Copyright of this thesis is held by the author,14 Sep 2012 12:29,Amélie Nothomb,Faculty of Arts and Humanities > Modern Langua...
2,Mental imaging as a psychotherapeutic tool: a ...,Studies have shown that mental imagery is nece...,Thesis (Doctoral),Doctor of Philosophy,1999,Copyright of this thesis is held by the author,13 Sep 2012 15:48,NaN,NaN
3,Identifying signal transduction components act...,"Traditionally, reactive oxygen species (ROS) h...",Thesis (Doctoral),Doctor of Philosophy,2007,Copyright of this thesis is held by the author,08 Sep 2011 18:34,NaN,NaN
4,Liturgical theology: children and the city,Liturgical Theology: Children and the City eng...,Thesis (Doctoral),Doctor of Philosophy,2003,Copyright of this thesis is held by the author,01 Aug 2012 11:36,NaN,NaN


Similarly, if we want to see the last few rows:

In [9]:
df.tail()

,Title,Abstract,Item Type,Award,Thesis Date,Copyright,Deposited On,Keywords,Faculty and Department
15,The Role of Story in Pastoral Theology: a theo...,Goals The thesis sets out to examine the value...,Thesis (Masters),Master of Letters,1992,Copyright of this thesis is held by the author,16 Nov 2012 10:59,NaN,NaN
16,Finite element modelling of transportation tun...,The aim of this thesis is to determine the gro...,Thesis (Doctoral),Doctor of Philosophy,1995,Copyright of this thesis is held by the author,09 Oct 2012 11:49,NaN,NaN
17,Settlement and economy in the forest and park ...,Decisive and far-reaching changes in upper Wea...,Thesis (Masters),Master of Arts,1979,Copyright of this thesis is held by the author,31 May 2012 09:26,NaN,Faculty of Social Sciences and Health > Geogra...
18,Mechanistic studies of S-nitrosothiol reaction...,A study of the reactions of various S-nitrosot...,Thesis (Doctoral),Doctor of Philosophy,1994,Copyright of this thesis is held by the author,24 Oct 2012 15:13,NaN,NaN
19,Poetry & sacrament: Being a commentary on the ...,"""The Kensington Mass"" was the last poem of the...",Thesis (Masters),Master of Letters,2002,Copyright of this thesis is held by the author,01 Aug 2012 11:40,NaN,NaN


To quickly understand the structure of the dataset (column names, number of rows, and data types), we can run:

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Title                   20 non-null     str  
 1   Abstract                20 non-null     str  
 2   Item Type               20 non-null     str  
 3   Award                   20 non-null     str  
 4   Thesis Date             20 non-null     str  
 5   Copyright               20 non-null     str  
 6   Deposited On            20 non-null     str  
 7   Keywords                1 non-null      str  
 8   Faculty and Department  3 non-null      str  
dtypes: str(9)
memory usage: 1.5 KB


This is especially useful when working with scraped or messy datasets where the data types may not be what we expect.
## Accessing Columns
We can access the column of a DataFrame by using its name:

In [11]:
df["Title"]

0     Runtime visualisation of object-oriented software
1     Redefining the Self:Life Writing, Fairy Tale a...
2     Mental imaging as a psychotherapeutic tool: a ...
3     Identifying signal transduction components act...
4            Liturgical theology: children and the city
5                        Reasoning biases and delusions
6     Guqin and Guzheng: the historical and contempo...
7     A comparative analysis of the wrist and ankle ...
8     Identification of excited states in conjugated...
9     A gulf cooperation council currency union: app...
10    Functionalised Pyridyl- and Pyrimidyl- Boronic...
11    Use of carbon isotope and C/N geochemistry in ...
12    'A local habitation and a name A Kristevan rea...
13    Assessing multi-version systems through fault ...
14    International instruments related to the preve...
15    The Role of Story in Pastoral Theology: a theo...
16    Finite element modelling of transportation tun...
17    Settlement and economy in the forest and p

This returns a Series, which is essentially a single column of data. For example, if we wanted to see all thesis titles:

In [12]:
titles = df["Title"]
titles

0     Runtime visualisation of object-oriented software
1     Redefining the Self:Life Writing, Fairy Tale a...
2     Mental imaging as a psychotherapeutic tool: a ...
3     Identifying signal transduction components act...
4            Liturgical theology: children and the city
5                        Reasoning biases and delusions
6     Guqin and Guzheng: the historical and contempo...
7     A comparative analysis of the wrist and ankle ...
8     Identification of excited states in conjugated...
9     A gulf cooperation council currency union: app...
10    Functionalised Pyridyl- and Pyrimidyl- Boronic...
11    Use of carbon isotope and C/N geochemistry in ...
12    'A local habitation and a name A Kristevan rea...
13    Assessing multi-version systems through fault ...
14    International instruments related to the preve...
15    The Role of Story in Pastoral Theology: a theo...
16    Finite element modelling of transportation tun...
17    Settlement and economy in the forest and p

Working with columns like this is a transferable skill because many data analysis tasks involve selecting and transforming particular columns.
## Filtering Data
A powerful feature of Pandas is the ability to filter rows based on conditions. For example, suppose we only want to see Master's theses:

In [13]:
masters = df[df["Item Type"] == "Thesis (Masters)"]
masters

,Title,Abstract,Item Type,Award,Thesis Date,Copyright,Deposited On,Keywords,Faculty and Department
1,"Redefining the Self:Life Writing, Fairy Tale a...",This thesis primarily focuses on Amélie Nothom...,Thesis (Masters),Master of Arts,2012,Copyright of this thesis is held by the author,14 Sep 2012 12:29,Amélie Nothomb,Faculty of Arts and Humanities > Modern Langua...
6,Guqin and Guzheng: the historical and contempo...,This thesis examines two Chinese musical instr...,Thesis (Masters),Master of Arts,1996,Copyright of this thesis is held by the author,13 Sep 2012 15:57,NaN,NaN
7,A comparative analysis of the wrist and ankle ...,There has been considerable debate concerning ...,Thesis (Masters),Master of Science,2001,Copyright of this thesis is held by the author,26 Jun 2012 15:23,NaN,NaN
11,Use of carbon isotope and C/N geochemistry in ...,The use of stable carbon isotope geochemistry ...,Thesis (Masters),Master of Science,2008,Copyright of this thesis is held by the author,27 Jul 2011 12:49,NaN,Faculty of Social Sciences and Health > Geogra...
13,Assessing multi-version systems through fault ...,Multi-version design (MVD) has been proposed a...,Thesis (Masters),Master of Science,2001,Copyright of this thesis is held by the author,26 Jun 2012 15:22,NaN,NaN
14,International instruments related to the preve...,The thesis gives a brief introduction into the...,Thesis (Masters),Master of Jurisprudence,2006,Copyright of this thesis is held by the author,09 Sep 2011 09:52,NaN,NaN
15,The Role of Story in Pastoral Theology: a theo...,Goals The thesis sets out to examine the value...,Thesis (Masters),Master of Letters,1992,Copyright of this thesis is held by the author,16 Nov 2012 10:59,NaN,NaN
17,Settlement and economy in the forest and park ...,Decisive and far-reaching changes in upper Wea...,Thesis (Masters),Master of Arts,1979,Copyright of this thesis is held by the author,31 May 2012 09:26,NaN,Faculty of Social Sciences and Health > Geogra...
19,Poetry & sacrament: Being a commentary on the ...,"""The Kensington Mass"" was the last poem of the...",Thesis (Masters),Master of Letters,2002,Copyright of this thesis is held by the author,01 Aug 2012 11:40,NaN,NaN


This creates a new DataFrame containing only rows that match the condition. Filtering like this is extremely common when analysing datasets.
## Selecting Columns
If we want to collect multiple columns, we can provide a list like so:

In [14]:
df[["Title", "Thesis Date"]]

,Title,Thesis Date
0,Runtime visualisation of object-oriented software,2003
1,"Redefining the Self:Life Writing, Fairy Tale a...",2012
2,Mental imaging as a psychotherapeutic tool: a ...,1999
3,Identifying signal transduction components act...,2007
4,Liturgical theology: children and the city,2003
5,Reasoning biases and delusions,1996
6,Guqin and Guzheng: the historical and contempo...,1996
7,A comparative analysis of the wrist and ankle ...,2001
8,Identification of excited states in conjugated...,2002
9,A gulf cooperation council currency union: app...,2006


## Basic Statistics
Pandas also makes it easy to compute basic statistics and summaries. For example, we can count how many theses exist for each item type:

In [15]:
df["Item Type"].value_counts()

Item Type
Thesis (Doctoral)       10
Thesis (Masters)         9
Thesis (Unspecified)     1
Name: count, dtype: int64

Similarly, we can count how many theses were awarded for each degree:

In [16]:
df["Award"].value_counts()

Award
Doctor of Philosophy       10
Master of Arts              3
Master of Science           3
Master of Letters           2
Unspecified                 1
Master of Jurisprudence     1
Name: count, dtype: int64

## Sorting Data
To sort theses by year:

In [17]:
df.sort_values("Thesis Date")

,Title,Abstract,Item Type,Award,Thesis Date,Copyright,Deposited On,Keywords,Faculty and Department
17,Settlement and economy in the forest and park ...,Decisive and far-reaching changes in upper Wea...,Thesis (Masters),Master of Arts,1979,Copyright of this thesis is held by the author,31 May 2012 09:26,NaN,Faculty of Social Sciences and Health > Geogra...
15,The Role of Story in Pastoral Theology: a theo...,Goals The thesis sets out to examine the value...,Thesis (Masters),Master of Letters,1992,Copyright of this thesis is held by the author,16 Nov 2012 10:59,NaN,NaN
18,Mechanistic studies of S-nitrosothiol reaction...,A study of the reactions of various S-nitrosot...,Thesis (Doctoral),Doctor of Philosophy,1994,Copyright of this thesis is held by the author,24 Oct 2012 15:13,NaN,NaN
16,Finite element modelling of transportation tun...,The aim of this thesis is to determine the gro...,Thesis (Doctoral),Doctor of Philosophy,1995,Copyright of this thesis is held by the author,09 Oct 2012 11:49,NaN,NaN
5,Reasoning biases and delusions,We know little about the formation and mainten...,Thesis (Doctoral),Doctor of Philosophy,1996,Copyright of this thesis is held by the author,09 Oct 2012 11:50,NaN,NaN
6,Guqin and Guzheng: the historical and contempo...,This thesis examines two Chinese musical instr...,Thesis (Masters),Master of Arts,1996,Copyright of this thesis is held by the author,13 Sep 2012 15:57,NaN,NaN
2,Mental imaging as a psychotherapeutic tool: a ...,Studies have shown that mental imagery is nece...,Thesis (Doctoral),Doctor of Philosophy,1999,Copyright of this thesis is held by the author,13 Sep 2012 15:48,NaN,NaN
7,A comparative analysis of the wrist and ankle ...,There has been considerable debate concerning ...,Thesis (Masters),Master of Science,2001,Copyright of this thesis is held by the author,26 Jun 2012 15:23,NaN,NaN
13,Assessing multi-version systems through fault ...,Multi-version design (MVD) has been proposed a...,Thesis (Masters),Master of Science,2001,Copyright of this thesis is held by the author,26 Jun 2012 15:22,NaN,NaN
19,Poetry & sacrament: Being a commentary on the ...,"""The Kensington Mass"" was the last poem of the...",Thesis (Masters),Master of Letters,2002,Copyright of this thesis is held by the author,01 Aug 2012 11:40,NaN,NaN


Or to sort from newest to oldest:

In [18]:
df.sort_values("Thesis Date", ascending=False)

,Title,Abstract,Item Type,Award,Thesis Date,Copyright,Deposited On,Keywords,Faculty and Department
1,"Redefining the Self:Life Writing, Fairy Tale a...",This thesis primarily focuses on Amélie Nothom...,Thesis (Masters),Master of Arts,2012,Copyright of this thesis is held by the author,14 Sep 2012 12:29,Amélie Nothomb,Faculty of Arts and Humanities > Modern Langua...
12,'A local habitation and a name A Kristevan rea...,This study is concerned with the concept of hu...,Thesis (Doctoral),Doctor of Philosophy,2008,Copyright of this thesis is held by the author,08 Sep 2011 18:24,NaN,NaN
11,Use of carbon isotope and C/N geochemistry in ...,The use of stable carbon isotope geochemistry ...,Thesis (Masters),Master of Science,2008,Copyright of this thesis is held by the author,27 Jul 2011 12:49,NaN,Faculty of Social Sciences and Health > Geogra...
3,Identifying signal transduction components act...,"Traditionally, reactive oxygen species (ROS) h...",Thesis (Doctoral),Doctor of Philosophy,2007,Copyright of this thesis is held by the author,08 Sep 2011 18:34,NaN,NaN
9,A gulf cooperation council currency union: app...,"The six states, that together comprise the Gul...",Thesis (Doctoral),Doctor of Philosophy,2006,Copyright of this thesis is held by the author,09 Sep 2011 09:56,NaN,NaN
14,International instruments related to the preve...,The thesis gives a brief introduction into the...,Thesis (Masters),Master of Jurisprudence,2006,Copyright of this thesis is held by the author,09 Sep 2011 09:52,NaN,NaN
10,Functionalised Pyridyl- and Pyrimidyl- Boronic...,The novel substituted pyridylboronic acids 2-e...,Thesis (Unspecified),Unspecified,2005,Copyright of this thesis is held by the author,09 Sep 2011 09:54,NaN,NaN
0,Runtime visualisation of object-oriented software,"Software is a complex and invisible entity, ye...",Thesis (Doctoral),Doctor of Philosophy,2003,Copyright of this thesis is held by the author,26 Jun 2012 15:21,NaN,NaN
4,Liturgical theology: children and the city,Liturgical Theology: Children and the City eng...,Thesis (Doctoral),Doctor of Philosophy,2003,Copyright of this thesis is held by the author,01 Aug 2012 11:36,NaN,NaN
8,Identification of excited states in conjugated...,This thesis reports quasi steady state photoin...,Thesis (Doctoral),Doctor of Philosophy,2002,Copyright of this thesis is held by the author,08 Sep 2011 18:23,NaN,NaN


# All Done!
That's all for now! You should now have a basic understanding of web scraping and Pandas. I hope this can assist you in your wonderful projects, and if you aren't using pandas or web scraping then I hope its been fun regardless.

This HackPack was made by Thomas Blair, the Head of Sponsorship at DUWiT Hacks 2026. If you have any questions, you can find me around most of the event, but I'll be running (and crushing) karaoke and serving midnight doughnuts!